# Module 00: GCP Environment, Authentication & Custom VPC Setup
Welcome to the JAX Multi-Node Training on GKE Learning Plan!

### Learning Objectives:
1. Install essential GCP, GKE, and JAX dependencies.
2. Authenticate Google Colab against Google Cloud (Application Default Credentials).
3. Load and inspect central environment variables (`PROJECT_ID`, `REGION`, `ZONE`, `CLUSTER_NAME`, etc.).
4. Provision custom VPC Network and Subnet with IP-Aliasing (`pods-range`, `services-range`).

## Install Essential Dependencies

In [ ]:
# Essential GCP, GKE, and JAX Multi-Node Dependencies
!pip install -U -q google-cloud-storage
!pip install -U -q google-cloud-container
!pip install -U -q google-auth
!pip install -U -q kubernetes
!pip install -U -q requests
!pip install -U -q jax jaxlib

# Optional: Vertex AI & Gemini SDKs (for Vertex AI tracking)
!pip install -U -q google-genai
!pip install -U -q google-cloud-aiplatform

## Restart the Colab Session

To use the newly installed packages in this Colab runtime, you must restart the runtime. You can do this by running the cell below, which restarts the current kernel.

The restart might take a minute or longer. After it's restarted, continue to the next step.

*Technically optional since Colab should prompt to restart the session after installing the dependencies above.*

In [ ]:
import IPython

app = IPython.Application.instance()
app.kernel.do_shutdown(True)

## Authenticate with Google Cloud (Colab Only)
Google Colab projects can authenticate against Google Cloud via the following calls. It is not required if running on Colab Enterprise or Vertex AI Workbench.

Running the cell below sets up the "[Application Default Credentials](https://cloud.google.com/docs/authentication/provide-credentials-adc)" which are used by our SDKs to automatically authenticate against Google Cloud Services.

In short, this is equivalent to the following gcloud CLI commands:
```bash
$ gcloud auth login
$ gcloud auth application-default login
```

An authentication pop-up will appear, please accept the permissions before proceeding.

### Alternative: Using Gemini API Keys
Some of these code samples will only work with a Google Cloud account, but the basic Gemini SDK will also work via an API key that can be obtained from [Google AI Studio](https://aistudio.google.com).

**Steps:**
1. Get an API key from: https://aistudio.google.com/app/apikey
2. Create a Colab "secret" called `AI_STUDIO_API_KEY` in the "Secrets" tab on the left hand side of Colab.
3. Make sure that `PROJECT_ID` *is not* defined. This ensures that AI Studio's API key will be used instead.

In [ ]:
from google.colab import userdata
from google.colab import auth
import sys

# Leave PROJECT_ID Blank to use an API Key instead
PROJECT_ID = "cloud-llm-preview1" # @param {type: "string"}
LOCATION = "us-central1" # @param {type: "string"}
# BUCKET = "" # @param{type:"string"}

if "google.colab" in sys.modules and PROJECT_ID != "":
    from google.colab import auth
    auth.authenticate_user(project_id=PROJECT_ID)

## Enable Google Cloud Services

In [ ]:
# !gcloud services enable \
#   cloudresourcemanager.googleapis.com \
#   aiplatform.googleapis.com \
#   documentai.googleapis.com \
#   notebooks.googleapis.com \
#   visionai.googleapis.com \
#   storage-component.googleapis.com \
#   cloudaicompanion.googleapis.com \
#   discoveryengine.googleapis.com \
#   --project {PROJECT_ID}

## Load Central Configuration Variables

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import config
cfg = config.load_config("../config.env")
config.print_config()

## Configure `gcloud` CLI & Active Project

In [ ]:
import os

PROJECT_ID = cfg.get("PROJECT_ID", "travolta-505921")
REGION = cfg.get("REGION", "us-central1")
ZONE = cfg.get("ZONE", "us-central1-a")
NETWORK_NAME = cfg.get("NETWORK_NAME", "jax-network")
SUBNET_NAME = cfg.get("SUBNET_NAME", "jax-subnet")

!gcloud config set project {PROJECT_ID}
!gcloud config set compute/region {REGION}
!gcloud config set compute/zone {ZONE}

In [ ]:
print("Enabling required GKE & Container APIs...")
!gcloud services enable \
    container.googleapis.com \
    artifactregistry.googleapis.com \
    cloudbuild.googleapis.com \
    iam.googleapis.com \
    compute.googleapis.com

## Configure IAM Permissions for Cloud Build Service Account

In [ ]:
# Grant Cloud Build & GCS Storage permissions to the Compute Engine Default Service Account
!PROJECT_NUMBER=$(gcloud projects describe {PROJECT_ID} --format="value(projectNumber)") && \
 gcloud projects add-iam-policy-binding {PROJECT_ID} \
    --member="serviceAccount:${PROJECT_NUMBER}-compute@developer.gserviceaccount.com" \
    --role="roles/storage.objectViewer" && \
 gcloud projects add-iam-policy-binding {PROJECT_ID} \
    --member="serviceAccount:${PROJECT_NUMBER}-compute@developer.gserviceaccount.com" \
    --role="roles/logs.writer" && \
 gcloud projects add-iam-policy-binding {PROJECT_ID} \
    --member="serviceAccount:${PROJECT_NUMBER}-compute@developer.gserviceaccount.com" \
    --role="roles/artifactregistry.writer"

## Provision Custom VPC & Subnet with IP Aliasing

In [ ]:
!gcloud compute networks create {NETWORK_NAME} --subnet-mode=custom

In [ ]:
!gcloud compute networks subnets create {SUBNET_NAME} \
    --network={NETWORK_NAME} \
    --region={REGION} \
    --range=10.0.0.0/20 \
    --secondary-range=pods-range=10.4.0.0/14,services-range=10.8.0.0/20

## Verify GCP Credentials & VPC Status

In [ ]:
!gcloud compute networks subnets describe {SUBNET_NAME} --region={REGION}